## cuDF API for GPU DataFrames

This notebook uses the NYC Yellow Taxi dataset to compare familiar pandas workflows with the GPU-native cuDF API.

In [1]:
import time
import pandas as pd
import cudf
import requests
from io import BytesIO

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## 1. Build GPU-native workflows with the cuDF API
If you're starting a new project and want to work directly with GPU DataFrames, cuDF is a great option. It mirrors the pandas API but executes operations on the GPU. Note that while very similar, some behaviors may differ from standard pandas.

Installation
In Google Colab, cuDF is pre-installed. For other environments, you can check your CUDA version and install the corresponding package:

```bash
!nvidia-smi
# Example for CUDA 13:
# pip install cudf-cu13
```

In [3]:
!nvidia-smi

Sun Aug 23 19:04:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.62                 Driver Version: 592.01         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5080 ...    On  |   00000000:02:00.0 Off |                  N/A |
| N/A   41C    P8              5W /   95W |     463MiB /  16303MiB |      9%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Loading data into a cuDF DataFrame

We'll use the first three months of 2023 NYC Taxi data, which gives us roughly 9-10 million rows.

In [4]:
def load_data_cudf(months=3):
    base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
    frames = []

    for month in range(1, months + 1):
        url = base_url.format(month)
        print(f"Downloading cuDF month {month:02d}...")
        response = requests.get(url)
        df = cudf.read_parquet(BytesIO(response.content))
        frames.append(df)
    return cudf.concat(frames, ignore_index=True)

def load_data_pandas(months=3):
    base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
    frames = []

    for month in range(1, months + 1):
        url = base_url.format(month)
        print(f"Downloading pandas month {month:02d}...")
        response = requests.get(url)
        df = pd.read_parquet(BytesIO(response.content))
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

df_gpu = load_data_cudf(months=3)
df_cpu = load_data_pandas(months=3)

print(f"Total rows (GPU): {len(df_gpu):,}")
print(f"Shape (GPU): {df_gpu.shape}")
print(f"Total rows (CPU): {len(df_cpu):,}")
print(f"Shape (CPU): {df_cpu.shape}")

Total rows (GPU): 9,384,487
Shape (GPU): (9384487, 20)
Total rows (CPU): 9,384,487
Shape (CPU): (9384487, 20)


## Working with cuDF

cuDF supports the same everyday operations that are available in pandas like selecting columns, filtering rows, creating new columns, and generating summary statistics.

In [5]:
# Selecting columns
trips = df_gpu[["PULocationID", "DOLocationID", "fare_amount", "trip_distance", "tip_amount"]]

# Filtering rows
high_fare = trips[trips["fare_amount"] > 20]

# Creating a new column
trips = trips.copy()
trips["tip_percentage"] = (trips["tip_amount"] / trips["fare_amount"] * 100)

# Summary statistics
trips.describe()

,PULocationID,DOLocationID,fare_amount,trip_distance,tip_amount,tip_percentage
count,9.384487e+06,9.384487e+06,9.384487e+06,9.384487e+06,9.384487e+06,9381178.0
mean,1.660118e+02,1.642366e+02,1.851788e+01,3.874278e+00,3.419354e+00,Inf
std,6.402818e+01,6.978675e+01,1.787964e+01,2.367626e+02,3.893056e+00,NaN
min,1.000000e+00,1.000000e+00,-9.599000e+02,0.000000e+00,-9.622000e+01,-455.6962025
25%,1.320000e+02,1.140000e+02,8.600000e+00,1.060000e+00,1.000000e+00,7.853403141
50%,1.620000e+02,1.620000e+02,1.280000e+01,1.790000e+00,2.800000e+00,23.98989899
75%,2.340000e+02,2.340000e+02,2.050000e+01,3.330000e+00,4.250000e+00,29.30232558
max,2.650000e+02,2.650000e+02,2.203100e+03,3.350043e+05,9.843000e+02,Inf


## Moving between pandas and cuDF

You'll sometimes need to move data between pandas and cuDF.

In [6]:
# pandas → cuDF
df_gpu_converted = cudf.from_pandas(df_cpu)
print(type(df_gpu_converted))

# cuDF → pandas
df_cpu_converted = df_gpu.to_pandas()
print(type(df_cpu_converted))

<class 'cudf.core.dataframe.DataFrame'>
<class 'pandas.DataFrame'>


## Benchmark: Grouped summary

We'll run the same function on both df_cpu (pandas) and df_gpu (cuDF) and use %time to measure how long each takes. This first example groups trips by pickup location and computes total fare, average distance, and trip count.

In [7]:
def summarize_trips(df):
    return (
        df.groupby("PULocationID")
          .agg(
              {
                  "fare_amount": "sum",
                  "trip_distance": "mean",
                  "passenger_count": "count",
              }
          )
          .sort_values("fare_amount", ascending=False)
    )

print("pandas:")
%time summary_pd = summarize_trips(df_cpu)

print("cuDF:")
%time summary_gpu = summarize_trips(df_gpu)

pandas:
CPU times: user 102 ms, sys: 38.5 ms, total: 141 ms
Wall time: 319 ms
cuDF:
CPU times: user 73.7 ms, sys: 5.84 ms, total: 79.5 ms
Wall time: 97.9 ms


## Benchmark: Chained operations

This example chains several steps together to find the most common drop-off location for every pickup location.

In [8]:
def run_chain(df):
    return (
        df[["PULocationID", "DOLocationID"]]
        .value_counts()
        .reset_index(name="count")
        .sort_values(
            ["PULocationID", "count"],
            ascending=[True, False],
        )
        .groupby("PULocationID")
        .head(1)
        .reset_index(drop=True)
    )

print("pandas:")
%time result_pd = run_chain(df_cpu)

print("cuDF:")
%time result_gpu = run_chain(df_gpu)

pandas:
CPU times: user 135 ms, sys: 102 ms, total: 237 ms
Wall time: 247 ms
cuDF:
CPU times: user 216 ms, sys: 23.8 ms, total: 240 ms
Wall time: 284 ms


The GPU advantage tends to compound as you chain more operations, since each step avoids a round-trip back to CPU memory.

In [9]:
def aggregate(df):
    return (
        df.groupby("PULocationID")
          .agg(
              {
                  "fare_amount": "sum",
                  "trip_distance": "sum",
                  "passenger_count": "count",
              }
          )
    )

print("pandas:")
%time agg_pd = aggregate(df_cpu)

print("cuDF:")
%time agg_gpu = aggregate(df_gpu)

pandas:
CPU times: user 99.4 ms, sys: 43.3 ms, total: 143 ms
Wall time: 142 ms
cuDF:
CPU times: user 55 ms, sys: 2.88 ms, total: 57.9 ms
Wall time: 65.5 ms
